# EXPORT_supabase_postgresql_FIXED

Exports final Databricks Delta tables from `workspace.default` to Supabase PostgreSQL.

This fixed version avoids Spark JDBC writes, which are blocked in Databricks Serverless/Free for JDBC DML. It still reads Databricks tables with Spark, but writes to Supabase using Python `psycopg`.

Security rule: credentials must be stored in Databricks Secrets, not hard-coded in this notebook.


## Before running

Create these Databricks Secrets:

- Scope: `supabase`
- Key: `jdbc_url`
- Key: `user`
- Key: `password`

Expected `jdbc_url` format for Supabase Session Pooler:

```text
jdbc:postgresql://aws-1-sa-east-1.pooler.supabase.com:5432/postgres?sslmode=require
```

Expected `user` format for Session Pooler:

```text
postgres.<project-ref>
```


In [0]:
%pip install psycopg[binary]

In [0]:
%restart_python

In [0]:
# =========================
# 1. Configuration
# =========================

SOURCE_DATABASE = "workspace.default"
TARGET_SCHEMA = "public"

SECRET_SCOPE = "supabase"
SECRET_KEY_JDBC_URL = "jdbc_url"
SECRET_KEY_USER = "user"
SECRET_KEY_PASSWORD = "password"

# Set to True only if you want to clean the Supabase tables before loading.
# Keep False for the first run if the target tables are empty.
TRUNCATE_TARGET_BEFORE_LOAD = False

# Set to True to create or replace PostgreSQL views after exporting the tables.
CREATE_CONSUMPTION_VIEWS = True

# Insert batch size used by psycopg.
BATCH_SIZE = 1000


In [0]:
# =========================
# 2. Read Supabase credentials from Databricks Secrets
# =========================

try:
    SUPABASE_JDBC_URL = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY_JDBC_URL)
    SUPABASE_USER = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY_USER)
    SUPABASE_PASSWORD = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY_PASSWORD)

    print("Supabase credentials loaded from Databricks Secrets.")
    print("JDBC URL loaded. Credentials are not displayed.")
except Exception as exc:
    raise RuntimeError(
        "Could not read Supabase credentials from Databricks Secrets. "
        "Create the secret scope and the keys: jdbc_url, user, password. "
        "Do not hard-code credentials in this notebook."
    ) from exc


In [0]:
# =========================
# 3. Helper functions
# =========================

from urllib.parse import urlparse, parse_qs
import pandas as pd
import numpy as np
import psycopg
from psycopg.rows import dict_row


def full_source_table(table_name: str) -> str:
    return f"{SOURCE_DATABASE}.{table_name}"


def quote_ident(identifier: str) -> str:
    return '"' + identifier.replace('"', '""') + '"'


def full_target_table(table_name: str) -> str:
    return f"{quote_ident(TARGET_SCHEMA)}.{quote_ident(table_name)}"


def parse_jdbc_url(jdbc_url: str) -> dict:
    """
    Converts:
    jdbc:postgresql://host:5432/postgres?sslmode=require
    into connection parameters for psycopg.
    """
    if not jdbc_url.startswith("jdbc:"):
        raise ValueError("Expected JDBC URL to start with 'jdbc:'.")

    parsed = urlparse(jdbc_url.replace("jdbc:", "", 1))
    query = parse_qs(parsed.query)

    return {
        "host": parsed.hostname,
        "port": parsed.port or 5432,
        "dbname": parsed.path.lstrip("/") or "postgres",
        "sslmode": query.get("sslmode", ["require"])[0],
    }


SUPABASE_CONN_INFO = parse_jdbc_url(SUPABASE_JDBC_URL)


def get_supabase_connection():
    return psycopg.connect(
        host=SUPABASE_CONN_INFO["host"],
        port=SUPABASE_CONN_INFO["port"],
        dbname=SUPABASE_CONN_INFO["dbname"],
        user=SUPABASE_USER,
        password=SUPABASE_PASSWORD,
        sslmode=SUPABASE_CONN_INFO["sslmode"],
        row_factory=dict_row,
    )


def execute_postgres_sql(sql: str, params: tuple | None = None):
    with get_supabase_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
        conn.commit()


def fetch_postgres_rows(sql: str, params: tuple | None = None):
    with get_supabase_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            return cur.fetchall()


def get_target_columns(table_name: str) -> list[str]:
    rows = fetch_postgres_rows(
        """
        select column_name
        from information_schema.columns
        where table_schema = %s
          and table_name = %s
        order by ordinal_position;
        """,
        (TARGET_SCHEMA, table_name),
    )

    columns = [row["column_name"] for row in rows]

    if not columns:
        raise ValueError(
            f"No columns found for target table {TARGET_SCHEMA}.{table_name}. "
            "Check whether the table exists in Supabase."
        )

    return columns


def align_dataframe_to_target(df, table_name: str):
    target_columns = get_target_columns(table_name)
    source_columns = df.columns

    selected_columns = [col for col in target_columns if col in source_columns]
    missing_columns = [col for col in target_columns if col not in source_columns]
    extra_columns = [col for col in source_columns if col not in target_columns]

    print(f"Target table: {TARGET_SCHEMA}.{table_name}")
    print(f"Selected columns: {selected_columns}")

    if missing_columns:
        print(f"Columns in target but not in source. PostgreSQL defaults/nulls will be used: {missing_columns}")

    if extra_columns:
        print(f"Columns in source but not in target. They will be ignored: {extra_columns}")

    if not selected_columns:
        raise ValueError(f"No matching columns found between source and target for table {table_name}.")

    return df.select(*selected_columns)


def clean_value(value):
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()

    return value


def write_dataframe_to_supabase(df, table_name: str):
    """
    Writes a Spark DataFrame to Supabase using psycopg.
    This avoids Spark JDBC write limitations in Databricks Free/Serverless.
    """
    target_table = full_target_table(table_name)
    pandas_df = df.toPandas()

    if pandas_df.empty:
        print(f"No rows to export for {target_table}.")
        return

    columns = list(pandas_df.columns)
    column_list = ", ".join(quote_ident(col) for col in columns)
    placeholders = ", ".join(["%s"] * len(columns))

    insert_sql = f"""
        insert into {target_table} ({column_list})
        values ({placeholders});
    """

    rows = [
        tuple(clean_value(value) for value in row)
        for row in pandas_df.itertuples(index=False, name=None)
    ]

    with get_supabase_connection() as conn:
        with conn.cursor() as cur:
            for start in range(0, len(rows), BATCH_SIZE):
                batch = rows[start:start + BATCH_SIZE]
                cur.executemany(insert_sql, batch)
        conn.commit()

    print(f"Inserted {len(rows)} rows into {target_table}.")


def count_supabase_rows(table_name: str) -> int:
    target_table = full_target_table(table_name)
    rows = fetch_postgres_rows(f"select count(*)::integer as row_count from {target_table};")
    return rows[0]["row_count"]


In [0]:
# =========================
# 4. Test Supabase connection
# =========================

test_table = "dim_tempo"

print(f"Testing connection using target table: {TARGET_SCHEMA}.{test_table}")

target_columns = get_target_columns(test_table)
row_count = count_supabase_rows(test_table)

print("Connection test succeeded.")
print(f"Target columns found in {test_table}: {target_columns}")
print(f"Current target row count in {test_table}: {row_count}")


In [0]:
# =========================
# 5. Tables to export
# =========================

# Order matters because facts depend on dimensions through foreign keys.
TABLES_TO_EXPORT = [
    "dim_tempo",
    "dim_doenca",
    "dim_bairro",
    "dim_sexo",
    "dim_bairro_geografica",
    "fato_arboviroses_final",
    "fato_covid_final",
    "fato_escorpiao_final",
    "fato_dengue_espacial_final",
    "fato_chikungunya_espacial_final",
]

print("Export order:")
for idx, table_name in enumerate(TABLES_TO_EXPORT, start=1):
    print(f"{idx}. {table_name}")


In [0]:
# =========================
# 6. Optional cleanup before reloading
# =========================

if TRUNCATE_TARGET_BEFORE_LOAD:
    print("Cleaning Supabase target tables before loading...")

    truncate_sql = (
        "truncate table "
        + ", ".join(full_target_table(table_name) for table_name in TABLES_TO_EXPORT)
        + " restart identity cascade;"
    )

    execute_postgres_sql(truncate_sql)
    print("Target tables truncated.")
else:
    print("TRUNCATE_TARGET_BEFORE_LOAD is False. Existing target data will be preserved.")


In [0]:
# =========================
# 7. Export Databricks tables to Supabase
# =========================

export_results = []

for table_name in TABLES_TO_EXPORT:
    print("=" * 80)
    print(f"Exporting: {table_name}")

    source_table = full_source_table(table_name)
    print(f"Reading source table: {source_table}")

    source_df = spark.table(source_table)
    source_count = source_df.count()

    print(f"Source rows: {source_count}")

    aligned_df = align_dataframe_to_target(source_df, table_name)

    write_dataframe_to_supabase(aligned_df, table_name)

    target_count = count_supabase_rows(table_name)

    export_results.append(
        {
            "table_name": table_name,
            "source_rows": source_count,
            "target_rows_after_export": target_count,
        }
    )

    print(f"Export finished: {table_name}")
    print(f"Target rows after export: {target_count}")

print("All exports finished.")


In [0]:
# =========================
# 8. Display export summary
# =========================

export_summary_df = spark.createDataFrame(export_results)
display(export_summary_df)


In [0]:
# =========================
# 9. Optional: create or replace consumption views in Supabase
# =========================

VIEW_SQL_STATEMENTS = [
    """
    create or replace view public.vw_serie_temporal as
    select
        'arboviroses'::text as fonte,
        dt.ano,
        dt.mes,
        dt.ordem_mes,
        dt.ano_mes,
        dd.doenca,
        fa.casos,
        fa.obitos
    from public.fato_arboviroses_final fa
    left join public.dim_tempo dt on fa.id_tempo = dt.id_tempo
    left join public.dim_doenca dd on fa.id_doenca = dd.id_doenca

    union all

    select
        'covid'::text as fonte,
        dt.ano,
        dt.mes,
        dt.ordem_mes,
        dt.ano_mes,
        dd.doenca,
        fc.casos,
        fc.obitos
    from public.fato_covid_final fc
    left join public.dim_tempo dt on fc.id_tempo = dt.id_tempo
    left join public.dim_doenca dd on fc.id_doenca = dd.id_doenca

    union all

    select
        'escorpiao'::text as fonte,
        dt.ano,
        dt.mes,
        dt.ordem_mes,
        dt.ano_mes,
        dd.doenca,
        fe.casos,
        fe.obitos
    from public.fato_escorpiao_final fe
    left join public.dim_tempo dt on fe.id_tempo = dt.id_tempo
    left join public.dim_doenca dd on fe.id_doenca = dd.id_doenca;
    """,

    """
    create or replace view public.vw_casos_espaciais as
    select
        'dengue'::text as fonte,
        dd.doenca,
        db.bairro,
        ds.sexo,
        dt.ano,
        dt.mes,
        dt.ordem_mes,
        dt.ano_mes,
        fd.data,
        fd.idade,
        fd.unidade_notificacao,
        fd.id_doenca,
        fd.id_bairro,
        fd.id_sexo,
        fd.id_tempo
    from public.fato_dengue_espacial_final fd
    left join public.dim_doenca dd on fd.id_doenca = dd.id_doenca
    left join public.dim_bairro db on fd.id_bairro = db.id_bairro
    left join public.dim_sexo ds on fd.id_sexo = ds.id_sexo
    left join public.dim_tempo dt on fd.id_tempo = dt.id_tempo

    union all

    select
        'chikungunya'::text as fonte,
        dd.doenca,
        db.bairro,
        ds.sexo,
        fch.ano_referencia as ano,
        null::text as mes,
        null::integer as ordem_mes,
        fch.ano_referencia::text as ano_mes,
        null::date as data,
        fch.idade,
        null::text as unidade_notificacao,
        fch.id_doenca,
        fch.id_bairro,
        fch.id_sexo,
        null::integer as id_tempo
    from public.fato_chikungunya_espacial_final fch
    left join public.dim_doenca dd on fch.id_doenca = dd.id_doenca
    left join public.dim_bairro db on fch.id_bairro = db.id_bairro
    left join public.dim_sexo ds on fch.id_sexo = ds.id_sexo;
    """,

    """
    create or replace view public.vw_casos_espaciais_geo as
    select
        ce.*,
        dbg.id_bairro_canonico,
        dbg.bairro_padronizado,
        dbg.latitude,
        dbg.longitude,
        dbg.latitude_plot,
        dbg.longitude_plot,
        dbg.is_mappable,
        dbg.spatial_status,
        dbg.spatial_quality,
        dbg.geocode_source
    from public.vw_casos_espaciais ce
    left join public.dim_bairro_geografica dbg
        on ce.id_bairro = dbg.id_bairro;
    """,

    """
    create or replace view public.vw_mapa_casos_bairro as
    select
        fonte,
        doenca,
        ano,
        mes,
        ordem_mes,
        ano_mes,
        bairro as bairro_original,
        id_bairro,
        id_bairro_canonico,
        bairro_padronizado,
        latitude,
        longitude,
        latitude_plot,
        longitude_plot,
        is_mappable,
        spatial_status,
        spatial_quality,
        geocode_source,
        count(*)::integer as casos
    from public.vw_casos_espaciais_geo
    group by
        fonte,
        doenca,
        ano,
        mes,
        ordem_mes,
        ano_mes,
        bairro,
        id_bairro,
        id_bairro_canonico,
        bairro_padronizado,
        latitude,
        longitude,
        latitude_plot,
        longitude_plot,
        is_mappable,
        spatial_status,
        spatial_quality,
        geocode_source;
    """,

    """
    create or replace view public.vw_mapa_casos_bairro_mapeavel as
    select
        fonte,
        doenca,
        ano,
        mes,
        ordem_mes,
        ano_mes,
        bairro_original,
        id_bairro,
        id_bairro_canonico,
        bairro_padronizado,
        latitude_plot as latitude,
        longitude_plot as longitude,
        casos
    from public.vw_mapa_casos_bairro
    where is_mappable = true
      and latitude_plot is not null
      and longitude_plot is not null;
    """,

    """
    create or replace view public.vw_qualidade_espacial as
    select
        fonte,
        spatial_status,
        spatial_quality,
        is_mappable,
        count(*)::integer as registros
    from public.vw_casos_espaciais_geo
    group by
        fonte,
        spatial_status,
        spatial_quality,
        is_mappable;
    """,
]

if CREATE_CONSUMPTION_VIEWS:
    print("Creating or replacing consumption views in Supabase...")

    for sql_statement in VIEW_SQL_STATEMENTS:
        execute_postgres_sql(sql_statement)
        first_line = sql_statement.strip().splitlines()[0]
        print(f"Executed: {first_line}")

    print("Consumption views created.")
else:
    print("CREATE_CONSUMPTION_VIEWS is False. Views were not created.")

In [0]:
# =========================
# 10. Validate key Supabase tables and views
# =========================

VALIDATION_OBJECTS = TABLES_TO_EXPORT + [
    "vw_serie_temporal",
    "vw_casos_espaciais_geo",
    "vw_mapa_casos_bairro_mapeavel",
    "vw_qualidade_espacial",
]

validation_results = []

for object_name in VALIDATION_OBJECTS:
    try:
        row_count = count_supabase_rows(object_name)
        validation_results.append(
            {
                "object_name": object_name,
                "row_count": row_count,
                "status": "ok",
            }
        )
    except Exception as exc:
        validation_results.append(
            {
                "object_name": object_name,
                "row_count": None,
                "status": f"error: {str(exc)[:200]}",
            }
        )

validation_df = spark.createDataFrame(validation_results)
display(validation_df)


## After running

Check Supabase:

1. Open **Table Editor**.
2. Confirm that dimensions and facts have rows.
3. Open **Database → Schema Visualizer** to see the PK/FK diagram.
4. In **SQL Editor**, test:

```sql
select * from vw_serie_temporal limit 10;
select * from vw_mapa_casos_bairro_mapeavel limit 10;
select * from vw_qualidade_espacial;
```

## GitHub safety checklist

Before committing:

- Do not include the Supabase password in the notebook.
- Do not include the full connection string with password.
- Keep this notebook using only `dbutils.secrets.get(...)`.
- Use a `.gitignore` for `.env`, local files, and exported credentials.
- Clear notebook outputs before exporting to GitHub.
